In [ ]:
# === Colab bootstrap (no-op outside Colab) ===========================
# Clones the repo, installs minimal deps, mounts Google Drive, and
# symlinks heavy assets from Drive into the paths the notebook uses.
# See COLAB_SETUP.md for details.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os, subprocess, sys
    REPO = "/content/INF8225_Projet"
    if not os.path.isdir(REPO):
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/Azcatchi17/INF8225_Projet.git", REPO,
        ])
    if os.getcwd() != REPO:
        os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from experiments.colab_setup import setup
    setup()
# ======================================================================


In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch
from skimage import io, transform
from tqdm import tqdm

# Imports Grounding DINO
from mmdet.apis import init_detector, inference_detector
from mmdet.utils import register_all_modules

# Imports MedSAM
from MedSAM.segment_anything import sam_model_registry
from MedSAM.MedSAM_Inference import medsam_inference

# ==========================================
# FONCTIONS UTILITAIRES
# ==========================================
def calculate_dice(mask_true, mask_pred):
    m_true = np.asarray(mask_true).astype(bool)
    m_pred = np.asarray(mask_pred).astype(bool)
    if m_true.sum() + m_pred.sum() == 0:
        return 1.0
    intersection = np.logical_and(m_true, m_pred).sum()
    return 2 * intersection / (m_true.sum() + m_pred.sum())

# ==========================================
# 1. INITIALISATION DES MODÈLES
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Appareil détecté : {device}")

register_all_modules()
dino_config = 'polyp_config_v2.py'
dino_checkpoint = 'work_dirs/polyp_config_v2/best_coco_bbox_mAP_epoch_5.pth'
dino_model = init_detector(dino_config, dino_checkpoint, device=device)

medsam_checkpoint = "MedSAM/work_dir/MedSAM/medsam_vit_b.pth"
medsam_model = sam_model_registry["vit_b"](checkpoint=medsam_checkpoint)
medsam_model = medsam_model.to(device)
medsam_model.eval()

# ==========================================
# 2. CONFIGURATION DES CHEMINS
# ==========================================
# Dossier où se trouvent TOUTES tes images et masques
base_img_folder = "data/Kvasir-SEG/images"
base_mask_folder = "data/Kvasir-SEG/masks"
test_json_path = "data/Kvasir-SEG/test.json"
output_folder = "data/outputs_medsam_dino"

os.makedirs(output_folder, exist_ok=True)

# Chargement du fichier test.json
with open(test_json_path, 'r') as f:
    test_data = json.load(f)

# On récupère la liste des images du test
test_images_list = test_data['images'] 

dice_list = []

print(f"\nDébut de l'évaluation sur les {len(test_images_list)} images du fichier test.json...")

# ==========================================
# 3. BOUCLE D'INFÉRENCE
# ==========================================
for img_info in tqdm(test_images_list, desc="Inférence Test Set"):
    file_name = img_info['file_name']
    img_id = img_info['id']
    
    img_path = os.path.join(base_img_folder, file_name)
    # On suppose que le masque porte le même nom que l'image
    mask_path = os.path.join(base_mask_folder, file_name)

    # Chargement Image & Masque GT
    if not os.path.exists(img_path):
        continue
        
    img_np = io.imread(img_path)
    if len(img_np.shape) == 2:
        img_3c = np.repeat(img_np[:, :, None], 3, axis=-1)
    else:
        img_3c = img_np
    H, W, _ = img_3c.shape

    true_seg = io.imread(mask_path)
    if len(true_seg.shape) == 3:
        true_seg = true_seg[:,:,0]
    true_seg = (true_seg > 0).astype(np.uint8)

    full_medsam_seg = np.zeros((H, W), dtype=np.uint8)

    with torch.no_grad():
        # A. DINO trouve la boîte
        result = inference_detector(dino_model, img_path, text_prompt="polyp.")
        pred_instances = result.pred_instances
        
        scores = pred_instances.scores.cpu().numpy()
        bboxes = pred_instances.bboxes.cpu().numpy()
        
        mask_conf = scores > 0.1 # Ton seuil de confiance
        valid_boxes = bboxes[mask_conf]
        valid_scores = scores[mask_conf]

        if len(valid_boxes) > 0:
            best_idx = np.argmax(valid_scores)
            best_box = valid_boxes[best_idx]
            
            # B. MedSAM segmente à partir de cette boîte
            # Pre-processing MedSAM
            img_1024 = transform.resize(img_3c, (1024, 1024), order=3, preserve_range=True, anti_aliasing=True).astype(np.uint8)
            img_1024 = (img_1024 - img_1024.min()) / np.clip(img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None)
            img_1024_tensor = torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(device)
            
            image_embedding = medsam_model.image_encoder(img_1024_tensor)
            
            box_np = np.array([best_box]) 
            box_1024 = box_np / np.array([W, H, W, H]) * 1024
            
            medsam_seg = medsam_inference(medsam_model, image_embedding, box_1024, H, W)
            full_medsam_seg[medsam_seg > 0] = 1

    # Calcul et stockage
    dice_score = calculate_dice(true_seg, full_medsam_seg)
    dice_list.append({
        "image_id": img_id,
        "file_name": file_name,
        "dice": dice_score
    })

# ==========================================
# 4. ANALYSE DES RÉSULTATS
# ==========================================
df = pd.DataFrame(dice_list)
df.to_csv("data/results/dice_final_report_v2.csv", index=False)

print("\n" + "="*40)
print(f"DICE MOYEN : {df['dice'].mean():.4f}")
print(f"DICE MEDIAN : {df['dice'].median():.4f}")
print(f"MEILLEUR DICE : {df['dice'].max():.4f}")
print(f"PIRE DICE : {df['dice'].min():.4f}")
print("="*40)

/home/paulb/INF8225/Projet/.venv/lib/python3.11/site-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
/home/paulb/INF8225/Projet/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


Appareil détecté : cuda


/home/paulb/INF8225/Projet/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loads checkpoint by local backend from path: work_dirs/polyp_config_v2/best_coco_bbox_mAP_epoch_5.pth


/home/paulb/INF8225/Projet/.venv/lib/python3.11/site-packages/mmengine/runner/checkpoint.py:347: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filena

...

noun_phrases: ['polyp']


Inférence Test Set:  99%|█████████▉| 99/100 [02:04<00:01,  1.34s/it][nltk_data] Downloading package punkt to ~/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     ~/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
/home/paulb/INF8225/Projet/.venv/lib/python3.11/site-packages/mmcv/cnn/bricks/transformer.py:524: UserWarning: position encoding of key ismissing in MultiheadAttention.
  warnings.warn(f'position encoding of key is'


noun_phrases: ['polyp']


Inférence Test Set: 100%|██████████| 100/100 [02:05<00:00,  1.25s/it]


DICE MOYEN : 0.8864
DICE MEDIAN : 0.9439
MEILLEUR DICE : 0.9832
PIRE DICE : 0.1968


In [3]:
df = pd.read_csv("data/results/dice_final_report_v2.csv")
df = df.sort_values(by="dice")
df.head(20)

,image_id,file_name,dice
74,75,cju410dnfl0960755y8lu8d79.jpg,0.196767
13,14,cju45rj7ln8980850a7821fov.jpg,0.355703
12,13,cju77q10sz9ug0801449wu1nu.jpg,0.409078
56,57,cju5waeduln160817w0agirve.jpg,0.446101
31,32,cju7et17a2vjk0755e743npl1.jpg,0.612378
93,94,cju3v56bwgy8v0871w14pz8fx.jpg,0.640403
78,79,cju7d2q1k27nf08715zshsckt.jpg,0.643121
84,85,cju84kplnl1y30755ropua1b0.jpg,0.647161
81,82,cju5vutu7ll8w0871dfp92n9p.jpg,0.688341
25,26,cju2rnkt22xep0801as160g9t.jpg,0.713170
